#

### 주제

북극 해빙예측 AI 경진대회

### 데이터

daily_train.csv : 학습용 일별 해빙 농도 데이터 정보

- date : 해빙 농도 관측일
- file_nm : 데이터 파일명

daily_train : 일별 해빙 농도 데이터

- 각 파일(*.npy)은 해빙 농도(0~250), 북극점(위성 관측 불가 영역), 해안선 마스크, 육지 마스크, 결측값 5개의 채널로 구성

weekly_train.csv : 학습용 주별 해빙 농도 데이터 정보

- weekly_start : 주별 해빙농도의 해당주 시작일
- week_file_nm : 데이터 파일명
- data_list : 주별 해빙 농도에 사용된 일별 해빙 농도 데이터

weekly_train : 주별 해빙 농도 데이터 / 월~일 일별 해빙농도의 평균

- 각 파일(*.npy)은 해빙 농도(0~250), 북극점(위성 관측 불가 영역), 해안선 마스크, 육지 마스크, 결측값 5개의 채널로 구성

public_daily_test.csv : public 기간 추론시 사용 가능한 일별 데이터 정보

- date : 해빙 농도 관측일
- file_nm : 데이터 파일명

public_weekly_test.csv : public 기간 추론시 사용 가능한 주별 데이터 정보

- weekly_start : 주별 해빙농도의 시작일
- week_file_nm : 데이터 파일명
- data_list : 주별 해빙 농도에 사용된 일별 해빙 농도 데이터

private_daily_test.csv : private 기간 추론시 사용 가능한 일별 데이터 정보

- date : 해빙 농도 관측일
- file_nm : 데이터 파일명

private_weekly_test.csv : private 기간 추론시 사용 가능한 주별 데이터 정보

- weekly_start : 주별 해빙농도의 시작일
- week_file_nm : 데이터 파일명
- data_list : 주별 해빙 농도에 사용된 일별 해빙 농도 데이터

sample_submission.csv : 정답제출 파일

### 코드 흐름

1. 라이브러리 및 데이터 로드
    - pandas, numpy, matplotlib
    - KNeighborRegressor
2. 이미지 데이터(.npy) 병합
    - 각 주차별 .npy 파일을 불러와 하나의 배열로 합침

In [ ]:
data = np.load('data/weekly_train/11158.npy')
data = data.reshape(1,448,304,5)

for i in dm.iloc[1:,0]:
    a = np.load('data/weekly_train/'+i)
    a = a.reshape(1,448,304,5)
    data = np.concatenate((data,a), axis=0)

data = data[:,:,:,0]

3. 월별 데이터셋 생성
    - week_of_month 기준으로 데이터 분리
    - 각 월 데이터를 기준으로 train1, train2, … 형태로 저장
    - globals()를 활용해 동적 변수 생성

In [ ]:
data = data.reshape(1016,1,448,304)

for i in dm.week_of_month.unique():
    temp_data = data[dm.week_of_month==i]
    temp = temp_data[0]
    for j in temp_data:
        temp = np.concatenate((temp, j), axis=0)
    globals()['train{}'.format(i)] = np.array(temp)

4. 평가 함수 정의
    - MAE
    - F1 score
    - MAE / F1
5. KNN 학습 구조
    - 과거 12개 주 데이터를 피처로 사용
    - 다음 1개 주를 타깃으로 설정
    - 이미지 전체를 flatten하여 벡터로 변환
    - transpose하여 pixel 단위 학습

In [ ]:
x_train = train1[6:18].reshape(12,-1).T
y_train = train1[18].reshape(1,-1).T
x_test  = train1[7:19].reshape(12,-1).T

6. K값 튜닝
    - 여러 차례 실험을 통해 n_neighbors 탐색
7. 예측 및 시각화
    - 예측 결과를 이미지 형태로 다시 복원
8. 제출 파일 생성

⇒ 과거 12주 해빙 이미지를 flatten하여 픽셀 단위 KNN 회귀를 수행하고, 대회 공식 점수(MAE/F1)에 맞게 K값을 튜닝하여 최종 제축 파일을 생성한 코드

### 새롭게 알게 된 내용 / 어려운 내용 / 배울 점

- KNN을 이미지 시계열 예측에 사용할 수 있다는 점을 알게 되었다. 보통 이미지 데이터는 CNN을 사용하는 경우가 많다고 생각했는데, 이 코드에서는 이미지를 flatten하여 픽셀 단위 회귀 문제로 변환한 뒤 KNN을 적용했다는 점이 인상적이었다.
- 픽셀 단위 학습 구조: 하나의 픽셀 위치가 하나의 학습 샘플이 되는 구조 → 기존에 내가 생각하던 데이터 구성 방식과 달라서 인상 깊었다.
- 보통 KNN에서는 작은 K값을 많이 사용하는데 이 코드에서는 600 이상의 매우 큰 값을 사용했다. 픽셀 단위 데이터 수가 매우 많고 노이즈를 줄이기 위해, 평균화 효과를 크게 주려고 큰 K값을 사용한 것으로 추측할 수 있었다.